# 05 — Test de la voix clonée XTTS-v2 (Colab)

**Objectif :** vérifier que XTTS-v2 produit une voix anglaise utilisable à partir d'un échantillon
de voix autorisé, et mesurer :

| Mesure | Pourquoi |
|---|---|
| temps de chargement et de génération | savoir si c'est utilisable sur CPU / GPU |
| **débit réel (caractères/s)** | remplacer les 14 car./s supposés dans l'adaptation au doublage |
| **CER** (Whisper relit l'audio généré) | mesurer l'intelligibilité (métrique demandée dans la répartition) |

La **similarité de voix** ne se mesure pas ici : elle sera notée à l'aveugle par 5 auditeurs.

**Avant de commencer :**
- l'échantillon de voix (10 à 30 s, voix seule, sans bruit) doit être **ta propre voix** ou une voix
  dont la personne a signé son accord ;
- le modèle fait environ **2,1 Go** (téléchargé dans Colab, pas sur le Mac) ; Whisper-small environ 1 Go ;
- fonctionne sur CPU (plus lent) ; un GPU T4 accélère si le quota le permet.

## 1. Environnement

In [ ]:
import sys, platform, torch
print("Python :", sys.version.split()[0], "|", platform.platform())
print("torch :", torch.__version__, "| GPU :", torch.cuda.is_available())
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

## 2. Installation de coqui-tts (fork maintenu de Coqui TTS, licence du code : MPL-2.0)

In [ ]:
!pip install -q coqui-tts
import TTS, transformers
print("coqui-tts :", TTS.__version__, "| transformers :", transformers.__version__)

## 3. Licence du modèle — à lire avant de continuer

Le **modèle** XTTS-v2 est sous *Coqui Public Model License* : https://coqui.ai/cpml.txt
- usage **non commercial uniquement** (projet académique : OK) ;
- toute personne qui reçoit une sortie du modèle doit aussi recevoir l'adresse de la licence.

Passe `I_ACCEPT_CPML` à `True` seulement après l'avoir lue.

In [ ]:
import os
I_ACCEPT_CPML = False
if not I_ACCEPT_CPML:
    raise RuntimeError("Lis https://coqui.ai/cpml.txt puis mets I_ACCEPT_CPML = True pour continuer.")
os.environ["COQUI_TOS_AGREED"] = "1"   # évite la question interactive de coqui-tts

## 4. Échantillon de voix de référence (avec consentement)

In [ ]:
import subprocess
CONSENT_CONFIRMED = False   # True seulement si la personne enregistrée a donné son accord écrit
if not CONSENT_CONFIRMED:
    raise RuntimeError("Confirme le consentement écrit de la personne dont la voix est clonée.")

from google.colab import files
uploaded = files.upload()               # wav, mp3, m4a... (enregistrement téléphone accepté)
raw_reference = list(uploaded)[0]
REFERENCE_WAV = "reference_voice.wav"
# Conversion en WAV mono : format le plus sûr pour XTTS.
subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", raw_reference, "-ac", "1", "-ar", "22050",
                REFERENCE_WAV], check=True)
import soundfile as sf
info = sf.info(REFERENCE_WAV)
print(f"Référence : {info.duration:.1f} s")
if not 6 <= info.duration <= 60:
    print("ATTENTION : 10 à 30 s de parole claire sont recommandées.")

## 5. Chargement du modèle

In [ ]:
import time
from TTS.api import TTS
start = time.perf_counter()
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(DEVICE)
LOAD_SECONDS = time.perf_counter() - start
print(f"Modèle chargé en {LOAD_SECONDS:.1f} s sur {DEVICE}")

## 6. Génération de phrases tests

Phrases anglaises représentatives du projet (sorties réelles du module de traduction sur la
vidéo de démonstration + phrases du jeu de test : nombres, noms propres, phrase longue).

In [ ]:
SENTENCES = [
    "Good morning, everybody.",
    "I'm going to introduce you to our automatic dubbing project.",
    "In order to maintain consistency, we keep the identifiers and timestamps of each segment.",
    "The result is quite satisfactory.",
    "The meeting is scheduled for March 15, 2025 at 2:30 p.m.",
    "Abdou Aziz is in charge of transcription and Baye Serigne of the voice.",
    "We compared four translation models on twenty-four test sentences.",
    "If the sentence is too long, the voice is slightly accelerated, but never more than twenty-five percent.",
]
os.makedirs("xtts_outputs", exist_ok=True)
results = []
for i, text in enumerate(SENTENCES, start=1):
    path = f"xtts_outputs/sentence_{i:02d}.wav"
    start = time.perf_counter()
    tts.tts_to_file(text=text, speaker_wav=REFERENCE_WAV, language="en", file_path=path)
    seconds = time.perf_counter() - start
    duration = sf.info(path).duration
    results.append({"id": i, "text": text, "audio_file": path, "audio_duration": round(duration, 3),
                    "generation_seconds": round(seconds, 2),
                    "real_time_factor": round(seconds / duration, 2),   # > 1 : plus lent que le temps réel
                    "chars_per_second": round(len(text) / duration, 2)})
    print(f"{i}: {duration:.2f} s d'audio en {seconds:.1f} s — {text}")

total_chars = sum(len(r["text"]) for r in results)
total_audio = sum(r["audio_duration"] for r in results)
print(f"\nDébit moyen : {total_chars / total_audio:.2f} caractères/s  (valeur supposée jusqu'ici : 14)")

### Écoute

In [ ]:
from IPython.display import Audio, display
for r in results:
    print(r["id"], r["text"]); display(Audio(r["audio_file"]))

## 7. Intelligibilité : CER avec Whisper

Whisper retranscrit chaque audio généré ; on compare au texte demandé.
**CER** = (substitutions + suppressions + insertions de caractères) / nombre de caractères du texte.
0 % = parfait. Limite : une erreur de Whisper compte comme une erreur de la voix.

In [ ]:
import re
from transformers import pipeline as hf_pipeline

asr = hf_pipeline("automatic-speech-recognition", model="openai/whisper-small",
                  device=0 if DEVICE == "cuda" else -1)


def normalize(text):
    # minuscules, sans ponctuation : on mesure les mots prononcés, pas la ponctuation
    return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9 ]", " ", text.lower())).strip()


def levenshtein(a, b):
    # distance d'édition classique (programmation dynamique)
    previous = list(range(len(b) + 1))
    for i, char_a in enumerate(a, start=1):
        current = [i]
        for j, char_b in enumerate(b, start=1):
            current.append(min(previous[j] + 1, current[j - 1] + 1, previous[j - 1] + (char_a != char_b)))
        previous = current
    return previous[-1]


for r in results:
    heard = asr(r["audio_file"], generate_kwargs={"language": "english"})["text"]
    reference, hypothesis = normalize(r["text"]), normalize(heard)
    r["whisper_text"] = heard.strip()
    r["cer"] = round(levenshtein(reference, hypothesis) / len(reference), 4)
    print(f"{r['id']}: CER {r['cer']:.1%} — {heard.strip()}")

errors = sum(levenshtein(normalize(r["text"]), normalize(r["whisper_text"])) for r in results)
print(f"\nCER global : {errors / sum(len(normalize(r['text'])) for r in results):.2%}")

## 8. Sauvegarde et téléchargement

In [ ]:
import csv, json, shutil
with open("05_xtts_results.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(results[0]))
    writer.writeheader(); writer.writerows(results)
run_info = {"model": "coqui/XTTS-v2", "coqui_tts": TTS.__version__, "torch": torch.__version__,
            "transformers": transformers.__version__, "device": DEVICE,
            "load_seconds": round(LOAD_SECONDS, 1), "reference_seconds": round(info.duration, 1),
            "mean_chars_per_second": round(total_chars / total_audio, 2), "asr_model": "openai/whisper-small"}
with open("05_run_info.json", "w", encoding="utf-8") as f:
    json.dump(run_info, f, ensure_ascii=False, indent=2)
shutil.make_archive("05_xtts_audio", "zip", "xtts_outputs")
for name in ("05_xtts_results.csv", "05_run_info.json", "05_xtts_audio.zip"):
    files.download(name)